In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_suppliers
# Source          : suppliers.csv
# Target          : procurement.bronze.bronze_suppliers
# Audit Table     : procurement.audit.duplicate_suppliers
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw suppliers master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load Suppliers master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Suppliers IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_SUPPLIERS)
print(AUDIT_DUPLICATE_SUPPLIERS)
print(SUPPLIERS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType,DataType,BooleanType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Suppliers schema
suppliers_schema = StructType([
    StructField("supplier_id", StringType(), False),
    StructField("supplier_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("contact_email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("supplier_rating",DecimalType(18,2),True),
    StructField("is_active", BooleanType(), True),
    StructField("onboarded_date", StringType(), True),
    StructField("payment_terms_default", StringType(), True),
])
# Read Employees master data from landing volume
bronze_suppliers_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(suppliers_schema)
    .load(SUPPLIERS_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_suppliers_df.count()}")

print("\nSchema:")
bronze_suppliers_df.printSchema()

print("\nColumns:")
print(bronze_suppliers_df.columns)

print("\nSampledata:")
display(bronze_suppliers_df.limit(10))

In [0]:
# Check the NULL and Blank supplier_ids
null_blank_supplier_id = bronze_suppliers_df .filter(col("supplier_id").isNull() | (trim(col("supplier_id")) == ""))

print(f"Total NULL or Blank supplier_ids : {null_blank_supplier_id.count()}")

display(null_blank_supplier_id)

In [0]:
#Check the NULL and Blank supplier name
null_blank_supplier_name = bronze_suppliers_df .filter(col("supplier_name").isNull() | (trim(col("supplier_name")) == ""))

print(f"Total NULL or Blank supplier_name : {null_blank_supplier_name.count()}")

display(null_blank_supplier_name)

In [0]:
#Check the NULL and Blank category
null_blank_category = bronze_suppliers_df.filter(col("category").isNull())

print(f"Total NULL or Blank category : {null_blank_category.count()}")

display(null_blank_category)


In [0]:
#Check the NULL and Blank country 
null_blank_country = bronze_suppliers_df.filter(col("country").isNull())

print(f"Total NULL or Blank country : {null_blank_country.count()}")

display(null_blank_country)


In [0]:
#Check the NULL and Blank payment_terms_default 
null_blank_payment_terms_default = bronze_suppliers_df.filter(col("payment_terms_default").isNull())

print(f"Total NULL or Blank payment_terms_default : {null_blank_payment_terms_default.count()}")

display(null_blank_payment_terms_default)


In [0]:
# ============================================================
# Identify Duplicate Suppliers IDs Ignore NULLs
# ============================================================

duplicate_supplier_keys = (
    bronze_suppliers_df
    .filter(
        col("supplier_id").isNotNull() &
        (trim(col("supplier_id")) != "")
    )
    .groupBy("supplier_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_supplier_keys)

In [0]:
# ============================================================
# Identify Duplicate Suppliers Records
# Business Rule: Keep the first occurrence of each Invoices ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("supplier_id").orderBy("onboarded_date")

supplier_rank_df = (
    bronze_suppliers_df
        .join(
            duplicate_supplier_keys.select("supplier_id"),
            on="supplier_id",
            how="inner"
        )
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)

display(supplier_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate Supplier Records
# ============================================================

duplicate_suppliers = (
    supplier_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate supplier Records : {duplicate_suppliers.count()}")

display(duplicate_suppliers)

In [0]:
# ============================================================
# Add Audit Metadata for Duplicate Supplier IDS
# ============================================================

from pyspark.sql.functions import current_timestamp, lit

duplicate_suppliers = (
    duplicate_suppliers
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("employees"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_suppliers)

In [0]:
# ============================================================
# Add Audit Metadata for null supplier IDs 
# ============================================================

invalid_suppliers = (
    null_blank_supplier_id
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Invoices"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(invalid_suppliers)

In [0]:
# ============================================================
# Write Duplicate Records to Audit Table
# ============================================================

duplicate_count = duplicate_suppliers.count()

if duplicate_count > 0:

    write_delta(
        df = duplicate_suppliers,
         table_name = AUDIT_DUPLICATE_SUPPLIERS
    )

    print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_SUPPLIERS}")

else:

    print("No duplicate Suppliers records found. Audit table not created.")

In [0]:
# ============================================================
# Write Invalid Supplier IDs to Audit Table
# ============================================================

invalid_count = invalid_suppliers.count()

if invalid_count > 0:

    write_delta(
        df = invalid_suppliers,
        table_name = AUDIT_INVALID_SUPPLIERS
    )

    print(f"Successfully written {invalid_count} invalid supplier record(s) to {AUDIT_INVALID_SUPPLIERS}")

else:

    print("No invalid invoice records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_suppliers_final_df = (
    bronze_suppliers_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("suppliers.csv"))
)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(
    df=bronze_suppliers_final_df,
    table_name=BRONZE_SUPPLIERS
)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_suppliers = spark.table(BRONZE_SUPPLIERS)

print(f"Total Bronze Records : {bronze_suppliers.count()}")

display(bronze_suppliers)

In [0]:
# ============================================================
# Bronze suppliers complete summary
# ============================================================
print("=" * 60)
print("Bronze Supplier Load Completed Successfully")
print("=" * 60)

print(f"{'Landing Records':<30}: {bronze_suppliers_df.count()}")

print(f"{'Duplicate Audit Records':<30}: {duplicate_suppliers.count()}")

print(f"{'Invalid Suppliers Records':<30}: {invalid_suppliers.count()}")

print(f"{'Total Audit Records':<30}: {duplicate_suppliers.count() + invalid_suppliers.count()}")

print(f"{'Bronze Records':<30}: {spark.table(BRONZE_SUPPLIERS).count()}")